# RSNA Knee: сабмит на Kaggle

Делает `submission.csv` из весов, обученных в `02_report_labels_mil.ipynb`. Ничего не учит.

## Почему сабмит отдельным ноутбуком

Это code competition: Kaggle сам перезапускает ноутбук на **скрытом** test, которого мы не видим. Поэтому test
нельзя подготовить заранее - ноутбук должен сам прочитать DICOM test и прогнать через модель. Internet при этом
выключен: всё нужное (веса, декодеры) подключается датасетами.

## Test готовится ровно так же, как обучающие данные

| Шаг | Обучение | Здесь |
|---|---|---|
| DICOM -> стопки срезов | `03_build_cache_kaggle.ipynb`: `pick_series`, `read_stack` | те же функции, в папку `test_series` |
| хранение | `studies/<ID>.npz` | такие же `<ID>.npz`, но во временной папке |
| окна и нормализация | `KneeStudyDataset` в 02 | тот же класс, режим проверки: окна фиксированные, без аугментаций |
| модель | `StudyMIL` | тот же класс, веса из `fold*.pth` |
| настройки (`IMG`, `CROP_MM`, `MAX_SLICES`, окна, энкодер) | конфиг 02 | читаются из файла весов |

Если хоть что-то готовить по-другому (другой кроп, другой порядок срезов), модель увидит «не те» картинки, и
качество упадёт без всякой ошибки. Поэтому код здесь буквально тот же.

## Ансамбль

Каждый файл `fold*.pth` - отдельная модель. Каждая смотрит на колено как есть и в зеркале, её предсказания
переводятся в **ранги** (место среди всех колен test, от 0 до 1), и ранги всех моделей усредняются.
Можно подключить несколько датасетов весов (например ViT-S и ViT-B): усреднятся все.

## Как запустить

1. Kaggle -> New Notebook -> File -> Import Notebook -> этот файл.
2. Input: соревнование `rsna-knee-abnormality-detection` и датасет(ы) с весами `fold*.pth`.
3. Желательно подключить датасет с декодерами сжатых DICOM (раздел 1).
4. Accelerator **GPU**, Internet **выключен**. Submit.

## Импорты

Как в билдере и в 02: `pydicom` и `cv2` для чтения DICOM, `torch` и `timm` для модели, `DataLoader` для батчей.
`DEV`: на Kaggle это видеокарта NVIDIA (`cuda`).

In [ ]:
import os
# сколько потоков могут занимать математические библиотеки; setdefault - только если не задано раньше
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "4")

import gc
import re
import math
import time
import glob
import json
import sys
import hashlib
import textwrap
import warnings
from copy import deepcopy
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor     # пул потоков: много серий читаются одновременно

import cv2                                             # ресайз картинок
import numpy as np
import pandas as pd
import pydicom                                         # чтение файлов .dcm
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader

warnings.filterwarnings("ignore")
T0 = time.time()

def log(msg):
    # печать с отметкой времени от старта: [   12.3s] сообщение
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)

try:
    display
except NameError:
    display = print

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")   # на Kaggle с GPU -> cuda
log(f"устройство: {DEV}, torch {torch.__version__}, timm {timm.__version__}")

## Настройки и веса

1. Пути Kaggle: данные соревнования и папка для `submission.csv`.
2. Поиск всех `fold*.pth` во всех подключённых датасетах (`os.walk`). Папки `train_series` и `test_series`
   пропускаем: в них сотни тысяч файлов, обход занял бы минуты.
3. Из **первого** файла весов читаем `config`: с какими настройками модель училась. Отсюда `IMG`, `CROP_MM`,
   `MAX_SLICES`, окна и размеры модели. Остальные файлы весов потом проверяются на совпадение.

| Параметр | Что значит |
|---|---|
| `SERIES_THREADS` | сколько серий читаем одновременно |
| `EVAL_BATCH` | колен в одном батче предсказания |
| `USE_AMP` | float16 на видеокарте: быстрее |
| `TTA_MIRROR` | смотреть на колено и на его зеркало |
| `WEIGHTS_PATH = False` | энкодер создаётся пустым: его веса придут из `fold*.pth` вместе с головой |

In [ ]:
import fnmatch                                          # сравнение имён с шаблоном "fold*.pth"

INPUT = Path("/kaggle/input")
DATA_DIR = INPUT / "competitions" / "rsna-knee-abnormality-detection"
if not DATA_DIR.exists():
    DATA_DIR = INPUT / "rsna-knee-abnormality-detection"
OUT_DIR = Path("/kaggle/working")

SERIES_THREADS = 24        # сколько серий читаем параллельно
EVAL_BATCH = 4
USE_AMP = DEV.type == "cuda"
AMP_DTYPE = torch.float16
TTA_MIRROR = True          # усредняем обычный и зеркальный вид колена
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]

# все fold*.pth из подключённых датасетов (в папки соревнования не заходим: там сотни тысяч файлов)
FOLD_FILES = []
for root, dirs, files in os.walk(INPUT, followlinks=True):
    # dirs[:] = ...: изменяем список подпапок на месте - os.walk не пойдёт в выкинутые
    dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
    FOLD_FILES += [Path(root) / f for f in files if fnmatch.fnmatch(f, "fold*.pth")]
FOLD_FILES = sorted(FOLD_FILES)
assert FOLD_FILES, "Нет весов fold*.pth: подключите датасет с весами в Input"

# настройки подготовки данных и модели берём из первого файла весов
CFG = torch.load(FOLD_FILES[0], map_location="cpu", weights_only=False)["config"]
IMG, CROP_MM, MAX_SLICES = CFG["IMG"], CFG["CROP_MM"], CFG["MAX_SLICES"]
K_WIN, WIN_BAND, FILL_MISSING_SLOTS = CFG["K_WIN"], tuple(CFG["WIN_BAND"]), CFG["FILL_MISSING_SLOTS"]
BACKBONE, UNFREEZE_LAST = CFG["BACKBONE"], CFG["UNFREEZE_LAST"]
HIDDEN, TF_DEPTH = CFG["HIDDEN"], CFG["TF_DEPTH"]
WEIGHTS_PATH = False       # предобученные веса не нужны: всё берётся из fold*.pth
for fp in FOLD_FILES:
    log(f"веса: {fp}")
log(f"настройки из весов: IMG={IMG}, CROP_MM={CROP_MM}, MAX_SLICES={MAX_SLICES}, K_WIN={K_WIN}, BACKBONE={BACKBONE}")

## 1. Декодеры сжатых DICOM

Часть DICOM-файлов хранится сжатой (JPEG, JPEG-2000, JPEG-LS), и сам `pydicom` их не распакует.
По словам автора ноутбука 0.958, в скрытом test такие файлы есть.

1. Ищем среди подключённых датасетов файлы `.whl` (готовые пакеты Python) с декодерами `pylibjpeg`, `gdcm`, `jpegls`.
2. Ставим их через `pip install --no-index` (без интернета, прямо из файлов).
3. Проверяем первые 100 серий test: сколько сжатых и сколько не прочиталось. Если `не прочитались` больше нуля,
   подключите датасет с декодерами: иначе эти серии будут пустыми, и колено получит 0.5.

In [ ]:
import importlib, subprocess

# все .whl на глубине 1-3 папок в Input, в имени которых есть название декодера
wheels = [w for pat in ("/kaggle/input/*/*.whl", "/kaggle/input/*/*/*.whl", "/kaggle/input/*/*/*/*.whl")
          for w in glob.glob(pat) if re.search(r"pylibjpeg|gdcm|jpegls", os.path.basename(w), re.I)]
if wheels:
    # --no-index: не ходить в интернет; --no-deps: не тянуть зависимости
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps"] + wheels, check=False)
    importlib.invalidate_caches()        # чтобы Python увидел только что установленные пакеты
log(f"колёс декодеров найдено: {len(wheels)}")

test_series = pd.read_csv(DATA_DIR / "test_series.csv")       # одна строка = одна серия test
n_fail = n_comp = 0
for r in test_series.head(100).itertuples():
    d = DATA_DIR / "test_series" / str(r.StudyInstanceUID) / str(r.SeriesInstanceUID)
    fs = [f for f in os.listdir(d) if f.endswith(".dcm")] if d.is_dir() else []
    if not fs:
        continue
    try:
        ds = pydicom.dcmread(d / fs[0])
        n_comp += ds.file_meta.TransferSyntaxUID.is_compressed   # True = файл сжат
        ds.pixel_array                                           # попытка распаковать картинку
    except Exception:
        n_fail += 1
log(f"проверка test: сжатых {n_comp}, не прочитались {n_fail}")

## 2. Dataset

Тот же класс `KneeStudyDataset`, что в `02_report_labels_mil.ipynb`, слово в слово. Записывается в файл
`knee_dataset.py` и импортируется. Здесь он работает в режиме проверки (`train=False`): окна фиксированные,
без зеркала и аугментаций. Подробное объяснение каждого шага - в 02, раздел 5.

In [ ]:
%%writefile knee_dataset.py
"""Датасет RSNA Knee: одно исследование (колено) -> один готовый пример для модели.

ВХОД: файл studies/<ID>.npz от билдера 03. Внутри словарь {"slot0": [n, 224, 224], "slot1": ...}:
по одной серии на ящик, все её срезы по порядку в теле, uint8 (яркость 0..255). Пустых ящиков в файле нет.

ВЫХОД __getitem__(i): x [6, K, 3, 224, 224], mask [6], pos [6, K], y [12], w [12].

Почему это отдельный файл, а не ячейка: DataLoader с num_workers > 0 на macOS запускает воркеры как НОВЫЕ
процессы Python (способ "spawn"). Новый процесс не видит классов из ячеек ноутбука и может только
импортировать модуль из файла. Поэтому ячейка %%writefile сначала пишет этот код в knee_dataset.py.
"""
import math
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

# ---------------- Шесть ящиков (слотов) ----------------
# (имя, плоскость, Fat_Suppression). Номер ящика = номер в списке: SAG_FS = 0, ..., AX_NOFS = 5.
# 6 = 3 плоскости (сбоку, спереди, сверху) x 2 режима (с подавлением жира и без).
SLOTS = [
    ("SAG_FS", "Sagittal", 1), ("COR_FS", "Coronal", 1), ("AX_FS", "Axial", 1),
    ("SAG_NOFS", "Sagittal", 0), ("COR_NOFS", "Coronal", 0), ("AX_NOFS", "Axial", 0),
]
N_SLOT = len(SLOTS)                                    # 6
SAG_SLOTS = [k for k, (_, plane, _) in enumerate(SLOTS) if plane == "Sagittal"]   # [0, 3]: для зеркала
FILL_PAIRS = ((0, 3), (1, 4), (2, 5))      # пары «та же плоскость, другой режим»: ими заполняем пустые ящики

# Среднее и разброс яркости по каналам R, G, B на ImageNet. DINOv2 учили на картинках, нормализованных так:
# x_norm = (x - mean) / std. Форма [1, 3, 1, 1], чтобы вычитать из батча картинок [N, 3, H, W] (broadcasting).
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
DEFAULT_AUG = dict(rot_deg=10.0, scale=0.10, shift=0.06, gamma=0.25, gain=0.10)   # если aug не передали


def window_centres(n, k, band, rng=None):
    """Центры k окон в стопке из n срезов. Окно = срезы [c-1, c, c+1], поэтому 1 <= c <= n-2.

    band = (0.08, 0.92): окна ставятся между 8% и 92% длины стопки (края - мягкие ткани, не сустав).
    rng=None (проверка): k точек равномерно, всегда одинаково.
    rng задан (обучение): полоса делится на k равных кусков, в каждом центр выбирается случайно.

    Пример, n = 22 (срезы 0..21), k = 4, проверка:
        a = 0.08 * 21 = 1.68, b = 0.92 * 21 = 19.32
        linspace(1.68, 19.32, 4) = [1.68, 7.56, 13.44, 19.32] -> округлили -> [2, 8, 13, 19]
    Тот же n, k = 2, обучение: куски [1.68, 10.5] и [10.5, 19.32], в каждом случайная точка, например [4, 15].
    """
    a = max(1.0, band[0] * (n - 1))          # левый край полосы; не меньше 1, иначе c-1 = -1 не существует
    b = min(n - 2.0, band[1] * (n - 1))      # правый край; не больше n-2, иначе c+1 = n не существует
    if b < a:                                # очень короткая стопка (3-4 среза): ставим всё в середину
        a = b = (n - 1) / 2
    if rng is None:
        c = np.linspace(a, b, k)             # k точек равномерно от a до b включительно
    else:
        edges = np.linspace(a, b, k + 1)     # границы k кусков: k + 1 точка
        # левая граница куска + случайная доля его ширины: rng.random(k) - k чисел из [0, 1)
        c = edges[:-1] + rng.random(k) * (edges[1:] - edges[:-1])
    # округляем до целого номера среза и ещё раз страхуемся, что c-1 и c+1 существуют
    return np.clip(np.round(c).astype(int), 1, max(1, n - 2))


def augment(x, rot_deg, scale, shift, gamma, gain, gen):
    """Случайные искажения, которые не меняют диагноз. Каждая картинка батча получает свои.

    x: float [N, 3, H, W] со значениями в [0, 1]. gen: torch.Generator (источник случайных чисел).
    Параметры (значения из конфига AUG):
        rot_deg = 10  -> поворот на случайный угол от -10 до +10 градусов
        scale = 0.10  -> зум ВНУТРЬ от 1.0 до 1.1 раза (наружу не зумим: по краям появилась бы чёрная рамка)
        shift = 0.06  -> сдвиг по x и по y до ±6% половины ширины картинки
        gamma = 0.25  -> контраст: x ** g, где g от e^-0.25 = 0.78 до e^0.25 = 1.28
        gain = 0.10   -> яркость: умножить на число от 0.9 до 1.1
    Флипов нет: вертикальный поменял бы местами бедро и голень, а «левое-правое» делает зеркало в __getitem__.
    """
    n = x.shape[0]                                           # сколько картинок
    r = lambda *s: torch.rand(*s, generator=gen)             # r(n) -> n случайных чисел из [0, 1)
    rot = (r(n) - 0.5) * 2 * math.radians(rot_deg)          # угол в радианах, от -rot_deg до +rot_deg
    sc = 1.0 + r(n) * scale                     # > 1 -> зум внутрь (края не дорисовываются)
    tx, ty = (r(n) - 0.5) * 2 * shift, (r(n) - 0.5) * 2 * shift   # сдвиги по x и y
    # Матрица аффинного преобразования theta [N, 2, 3] = [[cos, -sin, tx], [sin, cos, ty]].
    # Она говорит, ОТКУДА во входной картинке брать каждый пиксель выхода (координаты от -1 до 1).
    # Делим на sc: берём пиксели из области в sc раз меньше и растягиваем её на весь кадр = зум внутрь.
    cos, sin = torch.cos(rot) / sc, torch.sin(rot) / sc
    theta = torch.zeros(n, 2, 3)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = cos, -sin, tx
    theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = sin, cos, ty
    grid = F.affine_grid(theta, x.shape, align_corners=False)    # для каждого пикселя выхода: откуда его брать
    # grid_sample берёт пиксели по этой сетке: bilinear - смешивает 4 соседних, zeros - за краем чёрный
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    g = torch.exp((r(n, 1, 1, 1) - 0.5) * 2 * gamma)        # контраст (гамма): одно число на картинку
    k = 1.0 + (r(n, 1, 1, 1) - 0.5) * 2 * gain              # яркость
    # clamp_min(1e-4): 0 ** g безопасно не считать; clamp(0, 1): после умножения на k не выйти за [0, 1]
    return (x.clamp_min(1e-4) ** g * k).clamp(0, 1)


def mirror_batch(x, pos):
    """Зеркало колена для готового батча: TTA на проверке (модель смотрит на колено и на его отражение).

    x [B, S, K, 3, H, W], pos [B, S, K] -> то же самое, но для «зеркального» колена.
    Сагиттальные (сбоку): срезы идут от одного бока колена к другому. Зеркало = обратный порядок:
        flip(1) разворачивает порядок окон, flip(2) - порядок 3 срезов внутри окна, pos -> 1 - pos.
    Корональные и аксиальные: бок колена слева-справа на самой картинке -> flip(-1) по ширине.
    """
    x, pos = x.clone(), pos.clone()                  # копии, чтобы не испортить исходный батч
    for s in range(x.shape[1]):                      # по 6 ящикам
        if s in SAG_SLOTS:
            x[:, s] = x[:, s].flip(1).flip(2)
            pos[:, s] = 1 - pos[:, s].flip(-1)
        else:
            x[:, s] = x[:, s].flip(-1)
    return x, pos


class KneeStudyDataset(torch.utils.data.Dataset):
    """Один пример = одно исследование (колено).

    study_ids: ID исследований. i-й пример = study_ids[i].
    source: папка с файлами <ID>.npz ИЛИ словарь {ID: {"slot0": [n, H, W] uint8, ...}}.
    labels, weights: массивы [N, 12] в том же порядке, что study_ids (мягкие метки и их веса); None для test.
    train: True -> случайные окна, зеркало и аугментации; False -> фиксированные окна, без искажений.
    k_windows: сколько окон брать из каждого ящика (K).
    win_band: в какой полосе стопки ставить окна (доли от 0 до 1).
    fill_missing: заполнять пустые ящики парой из той же плоскости.
    mirror_p: вероятность зеркала при обучении.
    aug: словарь параметров augment (см. выше).
    """

    def __init__(self, study_ids, source, labels=None, weights=None, train=False, k_windows=4,
                 win_band=(0.08, 0.92), fill_missing=True, mirror_p=0.5, aug=None):
        # Здесь ничего не грузится с диска: только запоминаем настройки. Файлы открывает __getitem__.
        self.ids = [str(s) for s in study_ids]
        self.source = source
        self.labels = None if labels is None else np.asarray(labels, np.float32)
        self.weights = None if weights is None else np.asarray(weights, np.float32)
        self.train, self.k = train, k_windows
        self.win_band, self.fill_missing, self.mirror_p = tuple(win_band), fill_missing, mirror_p
        self.aug = DEFAULT_AUG if aug is None else aug

    def __len__(self):
        # DataLoader спрашивает, сколько всего примеров, чтобы знать, какие номера i можно просить
        return len(self.ids)

    def load_stacks(self, sid):
        # ID -> {номер ящика: uint8 [n_срезов, H, W]}. Например {0: [22,224,224], 1: [16,224,224], ...}
        if isinstance(self.source, dict):
            raw = self.source[sid]
        else:
            # with ... as z: файл .npz закроется сам после блока. z.files = ["slot0", "slot1", ...]
            with np.load(Path(self.source) / f"{sid}.npz") as z:
                raw = {key: z[key] for key in z.files}       # z[key] читает массив в память
        # "slot3" -> 3 (key[4:] = всё после "slot"); пустые массивы (если вдруг есть) пропускаем
        return {int(key[4:]): arr for key, arr in raw.items() if len(arr) > 0}

    def __getitem__(self, i):
        # DataLoader вызывает это для каждого номера i. Всё, что ниже, выполняется в воркере.
        sid = self.ids[i]
        stacks = self.load_stacks(sid)               # шаг 1: открыли файл колена
        h, w = next(iter(stacks.values())).shape[1:] if stacks else (224, 224)   # размер среза (224, 224)
        # Свои генераторы случайных чисел на каждый пример. torch.randint берёт число из генератора torch,
        # который DataLoader в каждом воркере засевает по-разному. Глобальный np.random в воркерах
        # получил бы одинаковое зерно, и все воркеры выбирали бы одинаковые «случайные» окна.
        seed = int(torch.randint(0, 2 ** 31 - 1, (1,)).item())
        rng, gen = np.random.default_rng(seed), torch.Generator().manual_seed(seed)

        # 1. Нет серии с fat-sat (или без) -> подставляем её пару из той же плоскости.
        #    Например, нет AX_NOFS (5), есть AX_FS (2): stacks[5] = stacks[2]. Если нет обеих, ящик пуст.
        if self.fill_missing:
            for a, b in FILL_PAIRS:
                if a not in stacks and b in stacks:
                    stacks[a] = stacks[b]
                elif b not in stacks and a in stacks:
                    stacks[b] = stacks[a]

        # 2. Зеркало колена (только обучение): левое колено превращается в «правое».
        #    Сагиттальная серия идёт поперёк колена -> разворачиваем порядок срезов;
        #    корональная и аксиальная -> отражаем каждый срез по горизонтали.
        #    s[::-1] - срезы в обратном порядке; s[:, :, ::-1] - каждый срез отражён по ширине.
        if self.train and rng.random() < self.mirror_p:
            stacks = {k: (s[::-1] if k in SAG_SLOTS else s[:, :, ::-1]) for k, s in stacks.items()}

        # 3. Окна: k центров в полосе стека, окно = срезы [c-1, c, c+1] -> 3 канала картинки.
        #    Сначала заготовки из нулей: пустой ящик так и останется нулями с mask = 0.
        x = np.zeros((N_SLOT, self.k, 3, h, w), np.uint8)     # [6, K, 3, 224, 224]
        mask = np.zeros(N_SLOT, np.float32)                   # [6]: 1 = ящик настоящий
        pos = np.zeros((N_SLOT, self.k), np.float32)          # [6, K]: где окно в стопке
        for k, s in stacks.items():
            n = len(s)                                         # сколько срезов в этой серии (12..32)
            c = window_centres(n, self.k, self.win_band, rng if self.train else None)   # например [2, 8, 13, 19]
            # np.stack([c-1, c, c+1], 1) -> номера срезов [K, 3], например [[1,2,3], [7,8,9], ...]
            # s[номера] -> [K, 3, H, W]: каждое окно = 3 соседних среза
            x[k] = s[np.stack([c - 1, c, c + 1], 1)]       # [k окон, 3 среза, H, W]
            mask[k] = 1.0
            pos[k] = c / max(n - 1, 1)                     # где окно в стеке: 0 = начало, 1 = конец

        # 4. В float [0, 1], аугментации (только обучение), нормализация ImageNet.
        #    view(N_SLOT * K, 3, h, w): все окна колена подряд, как батч обычных картинок RGB.
        t = torch.from_numpy(x).float().div_(255.0).view(N_SLOT * self.k, 3, h, w)
        if self.train:
            t = augment(t, gen=gen, **self.aug)               # **self.aug: rot_deg=..., scale=..., ...
        t = ((t - IMAGENET_MEAN) / IMAGENET_STD).view(N_SLOT, self.k, 3, h, w)   # обратно [6, K, 3, H, W]

        # метки и веса той же строки i; у test их нет -> нули и единицы (их всё равно никто не читает)
        y = self.labels[i] if self.labels is not None else np.zeros(12, np.float32)
        wt = self.weights[i] if self.weights is not None else np.ones(12, np.float32)
        return t, torch.from_numpy(mask), torch.from_numpy(pos), torch.from_numpy(y), torch.from_numpy(wt)

In [ ]:
sys.path.insert(0, str(Path.cwd()))      # чтобы Python нашёл только что записанный knee_dataset.py
from knee_dataset import KneeStudyDataset, mirror_batch, N_SLOT, SAG_SLOTS

## 3. Чтение test: те же функции, что в `03_build_cache_kaggle.ipynb`

Для каждого колена test: выбор серий по ящикам -> все срезы по порядку -> ориентация -> кроп 140 мм -> 224 px -> uint8.
Подробное объяснение каждого шага - в 03. Отличие одно: папка `test_series` вместо `train_series`.

In [ ]:
# Шесть ящиков: (имя, плоскость, Fat_Suppression). Номер ящика = номер в списке.
SLOTS = [  # (имя, плоскость, Fat_Suppression)
    ("SAG_FS", "Sagittal", 1), ("COR_FS", "Coronal", 1), ("AX_FS", "Axial", 1),
    ("SAG_NOFS", "Sagittal", 0), ("COR_NOFS", "Coronal", 0), ("AX_NOFS", "Axial", 0),
]


def count_files(d):
    # сколько .dcm в папке серии = сколько в ней срезов. os.scandir быстрее, чем glob, на тысячах папок
    try:
        return sum(1 for e in os.scandir(d) if e.name.endswith(".dcm"))
    except OSError:
        return 0                            # папки нет -> 0 срезов, такую серию не возьмём


def pick_series(series_df):
    # -> таблица «исследование, слот -> серия», одна строка на найденную комбинацию
    # series_df: строки train_series.csv (или test_series.csv): одна строка = одна серия
    df = series_df.copy()
    # путь к папке серии: <DATA_DIR>/test_series/<ID исследования>/<ID серии>
    df["dir"] = [str(DATA_DIR / "test_series" / str(a) / str(b))
                 for a, b in zip(df["StudyInstanceUID"], df["SeriesInstanceUID"])]
    # считаем файлы параллельно в SERIES_THREADS потоков: диск Kaggle медленный, потоки ждут его одновременно
    with ThreadPoolExecutor(SERIES_THREADS) as pool:
        df["n_files"] = list(pool.map(count_files, df["dir"]))
    rows = []
    for study, g in df.groupby("StudyInstanceUID"):          # g = все серии одного колена
        for k, (name, plane, fs) in enumerate(SLOTS):         # по 6 ящикам: k = 0..5
            # кандидаты в ящик k: нужная плоскость, нужный режим и хотя бы 3 среза (иначе это обзорный снимок)
            cand = g[(g["Anatomical_Plane"] == plane) & (g["Fat_Suppression"].astype(int) == fs)
                     & (g["n_files"] >= 3)]
            if len(cand):
                r = cand.sort_values("n_files", ascending=False).iloc[0]   # серия с наибольшим числом срезов
                rows.append({"StudyInstanceUID": study, "slot": k, "slot_name": name,
                             "SeriesInstanceUID": r["SeriesInstanceUID"], "dir": r["dir"],
                             "plane": plane, "n_files": int(r["n_files"])})
    return pd.DataFrame(rows)

In [ ]:
# Поля заголовка DICOM, которые нам нужны (читаем только их - так в разы быстрее):
HDR_TAGS = ["ImagePositionPatient", "ImageOrientationPatient", "PixelSpacing", "InstanceNumber"]
PLANES = ["Sagittal", "Coronal", "Axial"]
# Каноническая ориентация (из ноутбука 0.958). Для плоскости, у которой нормаль идёт вдоль оси
# 0 (x, лево-право) / 1 (y, перёд-зад) / 2 (z, верх-низ), задаём:
# (знак порядка срезов, (ось вдоль столбцов картинки, знак), (ось вдоль строк, знак))
# Система координат DICOM: x -> к ЛЕВОМУ боку пациента, y -> к спине, z -> к голове.
CANON = {
    0: (+1, (1, +1), (2, -1)),   # сагиттальная: слева направо по картинке = спереди назад, вниз = от головы
    1: (+1, (0, +1), (2, -1)),   # корональная: по картинке справа пациента налево, вниз = от головы
    2: (-1, (0, +1), (1, +1)),   # аксиальная: срезы сверху вниз, по картинке вниз = спереди назад
}


def _vec(v, n):
    # значение тега DICOM -> numpy-вектор из n чисел (или None, если тег пустой или битый)
    try:
        a = np.asarray([float(x) for x in v], float)
        return a if len(a) >= n else None
    except Exception:
        return None


def read_headers(d):
    # Шаг 1. Паспорта всех срезов серии БЕЗ картинок -> список словарей:
    # {"file": имя, "ipp": положение среза в теле (x,y,z мм), "iop": поворот (6 чисел),
    #  "ps": размер пикселя (мм по строкам и столбцам), "inst": номер среза от сканера}
    recs = []
    for f in sorted(os.listdir(d)):
        if not f.endswith(".dcm"):
            continue
        rec = {"file": f, "ipp": None, "iop": None, "ps": None, "inst": None}
        try:
            # stop_before_pixels=True: картинку не читаем; specific_tags: только нужные поля
            h = pydicom.dcmread(os.path.join(d, f), stop_before_pixels=True, force=True,
                                specific_tags=HDR_TAGS)
            rec["ipp"] = _vec(getattr(h, "ImagePositionPatient", []), 3)
            rec["iop"] = _vec(getattr(h, "ImageOrientationPatient", []), 6)
            rec["ps"] = _vec(getattr(h, "PixelSpacing", []), 2)
            rec["inst"] = float(getattr(h, "InstanceNumber", None) or 0)
        except Exception:
            pass                            # битый заголовок: срез останется, но без координат
        recs.append(rec)
    return recs


def sort_slices(recs, plane_name):
    # Шаг 2. Раскладываем срезы по порядку в теле.
    # -> (отсортированные записи, номер оси плоскости 0/1/2, IOP для ориентации)
    # iop = 6 чисел: первые 3 - куда идут строки картинки, вторые 3 - куда идут столбцы (векторы в теле).
    iop = next((r["iop"] for r in recs if r["iop"] is not None), None)
    if iop is not None:
        # нормаль = векторное произведение направлений строк и столбцов = направление «сквозь» срезы
        normal = np.cross(iop[:3], iop[3:])
        ax = int(np.abs(normal).argmax())   # вдоль какой оси тела режут: 0 сбоку, 1 спереди, 2 сверху
        if all(r["ipp"] is not None for r in recs):
            # ключ сортировки = проекция положения среза на нормаль = «насколько далеко срез вдоль нарезки».
            # sgn делает направление одинаковым у всех аппаратов (по таблице CANON)
            sgn = CANON[ax][0] * np.sign(normal[ax])
            recs = sorted(recs, key=lambda r: float(np.dot(r["ipp"], normal)) * sgn)
            return recs, ax, iop
    else:
        ax = PLANES.index(plane_name) if plane_name in PLANES else 0
    # запасной путь: координат нет -> по номеру среза от сканера, потом по имени файла
    return sorted(recs, key=lambda r: (r["inst"] or 0, r["file"])), ax, iop


def canon_orient(img, iop, ps, ax):
    # Шаг 5. Поворачиваем/отражаем срез к канонической ориентации; ps (размер пикселя) меняем вместе с ним
    if iop is None:
        return img, ps
    r, c = np.array(iop[:3]), np.array(iop[3:])
    _, (cax, csgn), (rax, rsgn) = CANON[ax]
    if abs(r[cax]) < abs(r[rax]):           # строки и столбцы перепутаны -> транспонируем
        img, r, c, ps = img.T, c, r, ps[::-1]
    if r[cax] * csgn < 0:
        img = img[:, ::-1]                  # отражаем по горизонтали
    if c[rax] * rsgn < 0:
        img = img[::-1]                     # отражаем по вертикали
    return np.ascontiguousarray(img), ps    # ascontiguousarray: после [::-1] массив «перевёрнут» в памяти


def crop_resize(img, ps):
    # Шаг 7. Квадрат CROP_MM мм вокруг центра -> IMG×IMG; если снимок меньше, дополняем нулями.
    # ps = (мм на пиксель по вертикали, по горизонтали). Пример: ps = 0.45 мм -> 140 мм = 311 пикселей.
    out = np.zeros((IMG, IMG), np.float32)
    mm_per_px = CROP_MM / IMG               # 140 / 224 = 0.625 мм в пикселе результата
    h, w = img.shape
    ch = min(h, int(round(CROP_MM / ps[0])))    # высота кропа в пикселях исходника (не больше картинки)
    cw = min(w, int(round(CROP_MM / ps[1])))    # ширина кропа
    y0, x0 = (h - ch) // 2, (w - cw) // 2       # левый верхний угол кропа: кроп по центру
    crop = np.ascontiguousarray(img[y0:y0 + ch, x0:x0 + cw])
    # размер кропа в пикселях результата: ch пикселей * ps мм / 0.625 мм (если снимок меньше 140 мм - меньше 224)
    nh = max(1, min(IMG, int(round(ch * ps[0] / mm_per_px))))
    nw = max(1, min(IMG, int(round(cw * ps[1] / mm_per_px))))
    interp = cv2.INTER_AREA if nh < ch else cv2.INTER_LINEAR   # AREA - без «лесенки» при уменьшении
    res = cv2.resize(crop, (nw, nh), interpolation=interp)     # cv2 принимает размер как (ширина, высота)
    oy, ox = (IMG - nh) // 2, (IMG - nw) // 2                   # кладём в центр чёрного квадрата
    out[oy:oy + nh, ox:ox + nw] = res
    return out


def decode(path):
    # Шаг 4. Файл .dcm -> картинка float32 [строки, столбцы] в «единицах сканера»
    ds = pydicom.dcmread(path, force=True)
    a = ds.pixel_array.astype(np.float32)
    if a.ndim == 3:                         # редкий «мультикадровый» файл - берём средний кадр
        a = a[len(a) // 2]
    # RescaleSlope/Intercept: перевод сырых чисел в единицы сканера (у МРТ обычно 1 и 0)
    a = a * float(getattr(ds, "RescaleSlope", 1) or 1) + float(getattr(ds, "RescaleIntercept", 0) or 0)
    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
        a = a.max() - a                     # в MONOCHROME1 «белое» и «чёрное» перевёрнуты
    return a


def read_stack(d, plane_name):
    # серия -> uint8 [n_срезов, IMG, IMG] по порядку в теле, или None, если ничего не прочиталось
    recs = read_headers(d)                                  # шаг 1: паспорта
    if not recs:
        return None
    recs, ax, iop = sort_slices(recs, plane_name)          # шаг 2: порядок
    n = len(recs)
    # шаг 3: не больше MAX_SLICES срезов, равномерно. n = 22 -> все 22; n = 320 -> 32 через ~10
    idx = np.unique(np.linspace(0, n - 1, min(n, MAX_SLICES)).round().astype(int))

    imgs = {}
    for i in idx:                                           # шаг 4: картинки
        try:
            imgs[int(i)] = decode(os.path.join(d, recs[int(i)]["file"]))
        except Exception:
            imgs[int(i)] = None
    ok = [i for i, a in imgs.items() if a is not None]
    if not ok:
        return None
    for i in imgs:                          # сломанный срез -> ближайший целый
        if imgs[i] is None:
            imgs[i] = imgs[min(ok, key=lambda j: abs(j - i))]

    ps_default = next((r["ps"] for r in recs if r["ps"] is not None), None)
    planes = []
    for i in idx:                                           # шаг 5: ориентация каждого среза
        a = imgs[int(i)]
        ps = recs[int(i)]["ps"] if recs[int(i)]["ps"] is not None else ps_default
        if ps is None:                      # нет размера пикселя -> считаем, что снимок = кропу
            ps = np.array([CROP_MM / max(a.shape)] * 2)
        planes.append(canon_orient(a, iop, list(ps), ax))

    # шаг 6: яркость. Порог «чёрного» и «белого» - перцентили 0.5% и 99.5% по ВСЕЙ серии
    # (a[::4, ::4] - каждый 4-й пиксель: для перцентилей хватает, и в 16 раз быстрее)
    allpx = np.concatenate([a[::4, ::4].ravel() for a, _ in planes])
    lo, hi = np.percentile(allpx, [0.5, 99.5])
    # (a - lo) / (hi - lo) -> 0..1 (выбросы обрезаются clip), затем шаг 7: кроп 140 мм и 224 px
    stack = np.stack([crop_resize(np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1), ps) for a, ps in planes])
    return (stack * 255).round().astype(np.uint8)          # шаг 8: 0..1 -> 0..255, 1 байт на пиксель

### Сборка test во временную папку

1. `pick_series(test_series)` раскладывает серии test по ящикам -> `TEST_INDEX`;
2. `build_study` для одного колена читает все его серии (`read_stack`) и пишет `TEST_DIR/<ID>.npz` - ровно как
   файлы кэша при обучении. Пустое колено (ничего не прочиталось) пропускается;
3. колена обрабатываются в 24 потока (`pool.map`).

Временная папка (`tempfile.gettempdir()`, на Kaggle это `/tmp`) не попадает в Output. Храним на диске, а не в памяти:
скрытый test может быть большим, а одно колено весит ~6 МБ.

`TEST_IDS` - колена, у которых есть хотя бы одна серия, в порядке `test.csv`.

In [ ]:
import tempfile

TEST_DIR = Path(tempfile.gettempdir()) / "test_studies"   # временная папка, не Output
TEST_DIR.mkdir(parents=True, exist_ok=True)
test_df = pd.read_csv(DATA_DIR / "test.csv")             # одна строка = одно колено test
TEST_INDEX = pick_series(test_series)
TEST_INDEX["StudyInstanceUID"] = TEST_INDEX["StudyInstanceUID"].astype(str)


def build_study(item):
    # одно исследование test -> TEST_DIR/<ID>.npz {"slot0": [n, IMG, IMG], ...}; None, если ничего не прочиталось
    # item = (ID колена, его строки TEST_INDEX) - так их отдаёт groupby
    sid, g = item
    stacks = {f"slot{r.slot}": read_stack(r.dir, r.plane) for r in g.itertuples(index=False)}
    stacks = {k: v for k, v in stacks.items() if v is not None}   # выкинуть серии, которые не прочитались
    if not stacks:
        return None
    np.savez(TEST_DIR / f"{sid}.npz", **stacks)      # без сжатия: файл временный, так быстрее
    return sid


with ThreadPoolExecutor(SERIES_THREADS) as pool:
    # set(...) - {None}: множество ID, для которых файл записан
    done = set(pool.map(build_study, TEST_INDEX.groupby("StudyInstanceUID"))) - {None}
TEST_IDS = [s for s in test_df["StudyInstanceUID"].astype(str) if s in done]
log(f"test: {len(test_df)} исследований, со снимками {len(TEST_IDS)}")

## 4. Модель и предсказание

`StudyMIL` и `predict` - те же, что в 02 (объяснение - там, разделы 6 и 7). Разница: `load_backbone` вызывается
с `weights=False`, то есть энкодер создаётся пустым, а все веса (энкодер и голова) приходят из `fold*.pth`.

In [ ]:
def load_backbone(name, weights_path, img):
    # Создаёт энкодер из библиотеки timm и загружает в него предобученные веса.
    # name: имя модели в timm, например "vit_small_patch14_dinov2.lvd142m"
    # weights_path: путь к .pth / None (скачать из интернета) / False (пустая сеть: веса придут из fold*.pth)
    # img: сторона входной картинки (224)
    kw = dict(num_classes=0)                # num_classes=0 -> сеть отдаёт вектор признаков, без головы
    if name.startswith(("vit_", "eva", "beit")):
        kw["img_size"] = img                # у ViT позиционные эмбеддинги зависят от размера входа
    if weights_path is False:
        return timm.create_model(name, pretrained=False, **kw)
    if weights_path and os.path.exists(weights_path):
        m = timm.create_model(name, pretrained=False, **kw)      # архитектура со случайными весами
        sd = torch.load(weights_path, map_location="cpu", weights_only=False)   # state_dict: {имя слоя: тензор}
        for key in ("state_dict", "model"):               # бывают обёртки {"model": {...}}
            if isinstance(sd, dict) and key in sd and isinstance(sd[key], dict):
                sd = sd[key]
        # убираем приставки, которые добавляют обёртки при сохранении ("module.blocks.0..." -> "blocks.0...")
        sd = {k.removeprefix("module.").removeprefix("model."): v for k, v in sd.items()}
        # выкидываем старую голову-классификатор: у нас своя голова, а энкодер без головы (num_classes=0)
        sd = {k: v for k, v in sd.items() if not k.startswith(("head.fc.", "fc.", "classifier."))
              and not (k.startswith("head.") and name.startswith("vit_"))}
        if "pos_embed" in sd and hasattr(m, "pos_embed") and sd["pos_embed"].shape != m.pos_embed.shape:
            # веса DINOv2 сохранены под 518px; под наш IMG интерполируем позиционные эмбеддинги
            # (518 / 14 = 37x37 кусочков-патчей, а у нас 224 / 14 = 16x16: «сетку мест» сжимаем до 16x16)
            from timm.layers import resample_abs_pos_embed
            sd["pos_embed"] = resample_abs_pos_embed(sd["pos_embed"], new_size=m.patch_embed.grid_size,
                                                     num_prefix_tokens=m.num_prefix_tokens)
        # strict=False + assert: сами проверяем, что ВСЕ ключи совпали. В бейзлайне 01 strict=False без проверки
        # молча пропустил бы несовпадение, и сеть училась бы со случайных весов.
        missing, unexpected = m.load_state_dict(sd, strict=False)
        assert not missing and not unexpected, (
            f"веса не подходят к {name}: missing={missing[:5]} unexpected={unexpected[:5]}")
        log(f"энкодер {name}: веса из файла, все ключи совпали")
        return m
    log(f"энкодер {name}: файла весов нет - скачиваю через timm (нужен Internet)")
    return timm.create_model(name, pretrained=True, **kw)


def set_trainable(bb, n_last):
    # всё заморозить, потом открыть «последнюю часть» сети
    # requires_grad = False: веса этого слоя не меняются при обучении (и не считается их градиент - быстрее)
    for p in bb.parameters():
        p.requires_grad = False
    if hasattr(bb, "blocks"):               # ViT: последние n_last блоков + финальная нормализация
        parts = list(bb.blocks)[-n_last:] + [getattr(bb, "norm", None)]
    elif hasattr(bb, "stages"):             # ConvNeXt и похожие CNN: две последние стадии + голова
        parts = list(bb.stages)[-2:] + [getattr(bb, "head", None), getattr(bb, "norm_pre", None)]
    else:
        parts = [bb]
    for part in parts:
        if part is not None:
            for p in part.parameters():
                p.requires_grad = True


class StudyMIL(nn.Module):
    # Модель «колено -> 12 логитов». Формы в комментариях даны для B = 8 колен, S = 6 ящиков, K = 2 окна.
    def __init__(self, backbone=BACKBONE, weights=WEIGHTS_PATH, unfreeze_last=UNFREEZE_LAST,
                 hidden=HIDDEN, tf_depth=TF_DEPTH):
        super().__init__()
        self.backbone = load_backbone(backbone, weights, IMG)     # DINOv2: картинка [3,224,224] -> вектор
        set_trainable(self.backbone, unfreeze_last)
        d = self.backbone.num_features                            # длина вектора энкодера: 384 у ViT-S
        # proj: вектор 384 -> вектор HIDDEN (256). Dropout(0.1) случайно зануляет 10% чисел при обучении
        # (против переобучения), LayerNorm выравнивает масштаб чисел.
        self.proj = nn.Sequential(nn.Dropout(0.1), nn.Linear(d, hidden), nn.LayerNorm(hidden))
        self.slot_emb = nn.Embedding(N_SLOT, hidden)            # «это сагиттальная FS» и т.п.
        # pos_emb: одно число pos (0..1) -> вектор 256 «где в колене это окно»
        self.pos_emb = nn.Sequential(nn.Linear(1, hidden), nn.GELU(), nn.Linear(hidden, hidden))
        if tf_depth:
            # 1 слой трансформера: каждое окно «смотрит» на все остальные окна колена.
            # hidden=256 - размер векторов, 4 - головы внимания, hidden*2 - ширина внутреннего слоя
            layer = nn.TransformerEncoderLayer(hidden, 4, hidden * 2, dropout=0.1,
                                               batch_first=True, norm_first=True)
            self.tf = nn.TransformerEncoder(layer, tf_depth, enable_nested_tensor=False)
        else:
            self.tf = None
        # внимание: для каждого окна 12 чисел «насколько это окно важно для патологии c»
        self.att = nn.Sequential(nn.Linear(hidden, 128), nn.Tanh(), nn.Dropout(0.2),
                                 nn.Linear(128, len(TARGETS)))
        self.cls_w = nn.Parameter(torch.randn(len(TARGETS), hidden) * 0.02)   # свой выход у каждой патологии
        self.cls_b = nn.Parameter(torch.zeros(len(TARGETS)))

    def forward(self, x, mask, pos):
        # x: float [B, S, K, 3, H, W] (уже нормализован в Dataset)   mask: [B, S]   pos: [B, S, K]   ->   логиты [B, 12]
        B, S, K = x.shape[:3]                                  # 8, 6, 2
        keep = mask.reshape(-1) > 0.5                          # какие (исследование, слот) есть
        feat = None
        if keep.any():
            # [8,6,2,3,224,224] -> [48,2,3,224,224] -> только настоящие ящики, например [46,2,...]
            # -> flatten(0, 1) -> [92, 3, 224, 224]: обычный батч картинок для энкодера
            xs = x.reshape(B * S, K, *x.shape[3:])[keep].flatten(0, 1)   # только существующие картинки
            f = self.backbone(xs)                              # [n_картинок, d]: [92, 384]
            feat = f.new_zeros(B * S, K, f.shape[-1])          # [48, 2, 384] нулей
            feat[keep] = f.view(-1, K, f.shape[-1])            # разложили векторы обратно по ящикам
        else:
            feat = x.new_zeros((B * S, K, self.backbone.num_features))
        h = self.proj(feat.view(B, S, K, -1))                  # [8, 6, 2, 256]
        # + «какой это ящик»: вектор ящика [1, 6, 1, 256] прибавляется ко всем окнам ящика
        h = h + self.slot_emb(torch.arange(S, device=x.device)).view(1, S, 1, -1)
        # + «где окно в колене»: pos [8, 6, 2] -> [8, 6, 2, 1] -> pos_emb -> [8, 6, 2, 256]
        h = h + self.pos_emb(pos.unsqueeze(-1).to(h.dtype))
        h = h.view(B, S * K, -1)                               # все окна исследования в одну «строку»: [8, 12, 256]
        pad = ~(mask > 0.5).unsqueeze(-1).expand(B, S, K).reshape(B, S * K)   # True = окна нет
        pad = pad & ~pad.all(1, keepdim=True)                  # исследование без серий: не маскируем всё
        if self.tf is not None:
            h = self.tf(h, src_key_padding_mask=pad)           # окна обменялись информацией; пустые не слушаем
        # att(h): [8, 12 окон, 12 патологий]. Пустым окнам -1e4, чтобы после softmax их вес был 0.
        # softmax(1): по окнам. Для каждой патологии веса 12 окон в сумме дают 1.
        a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)   # [B, окна, 12]
        # для каждой патологии c: взвешенная сумма векторов окон с её весами -> [8, 12 патологий, 256]
        pooled = torch.einsum("btc,btd->bcd", a.to(h.dtype), h)                   # [B, 12, hidden]
        # у каждой патологии свой линейный классификатор: скалярное произведение с cls_w[c] + cls_b[c]
        return (pooled * self.cls_w).sum(-1) + self.cls_b

In [ ]:
@torch.no_grad()
def predict(model, loader, tta_mirror=TTA_MIRROR):
    # -> вероятности [число исследований в loader, 12] в порядке loader (shuffle=False)
    # @torch.no_grad(): градиенты не нужны -> меньше памяти и быстрее
    model.eval()                 # выключает Dropout: на проверке модель должна отвечать одинаково
    out = []
    for x, m, p, _, _ in loader:            # метки и веса (_, _) здесь не нужны
        x, m, p = x.to(DEV), m.to(DEV), p.to(DEV)
        with torch.autocast(DEV.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            prob = torch.sigmoid(model(x, m, p).float())     # логиты -> вероятности 0..1
            if tta_mirror:                                 # второй взгляд: зеркальное колено
                xm, pm = mirror_batch(x, p)
                prob = 0.5 * (prob + torch.sigmoid(model(xm, m, pm).float()))
        out.append(prob.cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, 12), np.float32)

## 5. Сабмит

1. Сразу пишем `submission.csv` со всеми 0.5: если дальше что-то упадёт, на диске всё равно будет валидный сабмит
   (с AUC 0.5, но без ошибки «нет файла»).
2. Один `DataLoader` по всем колёнам test (`num_workers=0`: на Kaggle данные уже на быстром диске, воркеры не нужны).
3. Для каждого файла весов: проверить, что настройки совпадают с первым файлом; собрать модель с размерами из его
   `config`; загрузить веса; предсказать (как есть + зеркало); перевести в ранги.
4. Ранги всех моделей усредняются. Склеиваем с `test.csv` по `StudyInstanceUID` (`merge(how="left")`: все колена
   test остаются, даже без снимков). Колено без снимков получает 0.5.

In [ ]:
bench = test_df[["StudyInstanceUID"]].copy()
for t in TARGETS:
    bench[t] = 0.5
bench.to_csv(OUT_DIR / "submission.csv", index=False)    # запасной валидный сабмит

loader = DataLoader(KneeStudyDataset(TEST_IDS, TEST_DIR, train=False, k_windows=K_WIN, win_band=WIN_BAND,
                                     fill_missing=FILL_MISSING_SLOTS),
                    batch_size=EVAL_BATCH, shuffle=False, num_workers=0)
ranks = []
for fp in FOLD_FILES:
    ck = torch.load(fp, map_location="cpu", weights_only=False)   # {"model": веса, "config": настройки}
    cfg = ck["config"]
    assert (cfg["IMG"], cfg["CROP_MM"], cfg["K_WIN"]) == (IMG, CROP_MM, K_WIN), f"{fp.name} обучен с другими настройками: {cfg}"
    model = StudyMIL(backbone=cfg["BACKBONE"], weights=False, unfreeze_last=cfg["UNFREEZE_LAST"],
                     hidden=cfg["HIDDEN"], tf_depth=cfg["TF_DEPTH"]).to(DEV)
    model.load_state_dict(ck["model"])                  # strict по умолчанию: все ключи должны совпасть
    pr = predict(model, loader)                         # [n_test, 12] вероятности
    ranks.append(pd.DataFrame(pr).rank(pct=True).values)   # -> ранги 0..1 по каждой цели
    log(f"{fp.name}: предсказано {len(TEST_IDS)} исследований")
    del model, ck
    gc.collect()

sub = pd.DataFrame(np.mean(ranks, axis=0), columns=TARGETS)   # среднее рангов всех моделей
sub.insert(0, "StudyInstanceUID", TEST_IDS)
sub = test_df[["StudyInstanceUID"]].astype(str).merge(sub, on="StudyInstanceUID", how="left")
sub[TARGETS] = sub[TARGETS].fillna(0.5)                 # колена без снимков
sub.to_csv(OUT_DIR / "submission.csv", index=False)
log(f"submission.csv: {sub.shape}, пустых значений: {int(sub[TARGETS].isna().sum().sum())}")
display(sub.head())